In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objs as go
import urllib.request
import plotly.io as pio

pio.renderers.default = "browser"  # ← Força abrir no navegador

def carregar_dados(url):
    context = urllib.request.ssl._create_unverified_context()  # Desativa a verificação
    with urllib.request.urlopen(url, context=context) as response:
        df = pd.read_csv(response)
    df.rename(columns={
        '% year': 'year', ' month': 'month', ' day': 'day',
        ' hour': 'hour', ' minute': 'minute', ' second (GMT/UTC)': 'second',
        ' water level (meters)': 'water_level(m)'
    }, inplace=True)
    df['datetime'] = pd.to_datetime(df[['year', 'month', 'day', 'hour', 'minute', 'second']])
    return df

# Dicionário com as estações e URLs
estacoes_estrela = {
    "EST1 - Foz do Boa Vista": "https://app.tidesatglobal.com/est1/est1_out.csv",
    "EST2 - Porto de Estrela": "https://app.tidesatglobal.com/est2/est2_out.csv"
}

# Lista de traços para o gráfico
tracos = []

for nome_estacao, url in estacoes_estrela.items():
    try:
        df = carregar_dados(url)
        tracos.append(go.Scatter(
            x=np.array(df["datetime"]),
            y=df["water_level(m)"],
            mode='lines',
            name=nome_estacao
        ))
    except Exception as e:
        print(f"Erro ao carregar {nome_estacao}: {e}")

# Criar e exibir o gráfico
layout = go.Layout(
    title="Sobreposição de Nível d'Água - Estações de Estrela",
    xaxis_title="Data/Hora",
    yaxis_title="Nível (m)",
    height=600
)

fig = go.Figure(data=tracos, layout=layout)
fig.show()  # Agora abrirá em uma nova aba do seu navegador padrão